In [196]:
import os

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
from torchvision.transforms import v2

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [197]:
class SimpsonsDataset(Dataset):
    def __init__(self, path, transform=None):
        self.path = path
        self.transform = transform

        self.classes = sorted(
            [d for d in os.listdir(self.path) if os.path.isdir(os.path.join(self.path, d))]
        )
        
        self.class_to_idx = {class_name: i for i, class_name in enumerate(self.classes)}

        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(self.path, class_name)

            for file_name in sorted(os.listdir(class_dir)):
                file_path = os.path.join(class_dir, file_name)

                if os.path.isfile(file_path):
                    self.samples.append((file_path, self.class_to_idx[class_name]))

    def __getitem__(self, index):
        image_path, label = self.samples[index]

        img = read_image(str(image_path))
        if self.transform is not None:
            img = self.transform(img)

        return img, label

    def __len__(self):
        return len(self.samples)

In [198]:
train_transform = v2.Compose([
    v2.RandomResizedCrop((224, 224), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=10),
    v2.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]),
])

val_transform = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]),
])

In [199]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

PATH = 'data/simpsons_dataset'
SEED = 42
RARE_THRESHOLD  = 5

full_train_dataset = SimpsonsDataset(path=PATH, transform=train_transform)
full_val_dataset = SimpsonsDataset(path=PATH, transform=val_transform)

labels = [label for _, label in full_train_dataset.samples]

class_counts = Counter(labels)
rare_classes = {cls for cls, count in class_counts.items() if count < RARE_THRESHOLD}

rare_indices = []
normal_indices = []

for i, label in enumerate(labels):
    if label in rare_classes:
        rare_indices.append(i)
    else:
        normal_indices.append(i)

normal_labels = [labels[i] for i in normal_indices]

train_idx_norm, temp_idx_norm = train_test_split(
    normal_indices,
    test_size=0.2,
    stratify=normal_labels,
    random_state=SEED
)

temp_labels = [labels[i] for i in temp_idx_norm]

val_idx_norm, test_idx_norm = train_test_split(
    temp_idx_norm,
    test_size=0.5,
    stratify=temp_labels,
    random_state=SEED
)

train_idx = list(train_idx_norm) + rare_indices
val_idx   = list(val_idx_norm)
test_idx  = list(test_idx_norm)

In [200]:
from collections import Counter

print("Train:", Counter([labels[i] for i in train_idx]))
print("Val:",   Counter([labels[i] for i in val_idx]))
print("Test:",  Counter([labels[i] for i in test_idx]))

Train: Counter({15: 1797, 28: 1163, 27: 1162, 20: 1083, 4: 1074, 22: 1033, 17: 965, 32: 955, 6: 954, 25: 863, 7: 789, 0: 730, 37: 702, 2: 498, 16: 398, 9: 375, 11: 366, 29: 286, 18: 248, 24: 197, 41: 145, 21: 102, 14: 97, 3: 85, 36: 82, 5: 78, 35: 71, 31: 58, 23: 57, 33: 52, 39: 44, 8: 38, 34: 36, 1: 33, 38: 32, 30: 26, 13: 22, 12: 22, 26: 14, 10: 6, 40: 6, 19: 3})
Val: Counter({15: 224, 28: 145, 27: 145, 20: 135, 4: 134, 22: 129, 17: 121, 32: 120, 6: 119, 25: 108, 7: 99, 0: 92, 37: 87, 2: 62, 16: 50, 9: 47, 11: 45, 29: 36, 18: 31, 24: 24, 41: 18, 21: 13, 14: 12, 3: 11, 36: 11, 5: 10, 35: 9, 33: 7, 23: 7, 31: 7, 34: 5, 1: 5, 39: 5, 38: 4, 8: 4, 30: 3, 12: 3, 26: 2, 13: 2, 40: 1, 10: 1})
Test: Counter({15: 225, 28: 146, 27: 145, 20: 136, 4: 134, 22: 129, 17: 120, 6: 120, 32: 119, 25: 108, 7: 98, 0: 91, 37: 88, 2: 63, 16: 50, 9: 47, 11: 46, 29: 36, 18: 31, 24: 25, 41: 18, 21: 13, 14: 12, 36: 10, 5: 10, 3: 10, 35: 9, 31: 7, 23: 7, 33: 6, 39: 6, 8: 5, 38: 4, 34: 4, 1: 4, 30: 3, 13: 3, 12: 

In [201]:
from torch.utils.data import Subset

train_dataset = Subset(full_train_dataset, train_idx)
val_dataset   = Subset(full_val_dataset, val_idx)
test_dataset  = Subset(full_val_dataset, test_idx)

In [202]:
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [203]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = [labels[i] for i in train_idx]
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

In [204]:
import torch.nn as nn
from torchvision import models

num_classes = len(full_train_dataset.classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [205]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples

    return epoch_loss, epoch_acc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)
        
    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples

    return epoch_loss, epoch_acc

In [206]:
NUM_EPOCHS = 10

best_val_acc = 0.0
best_state = None

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

Epoch 1/10 | train_loss=3.1611 | train_acc=0.2360 | val_loss=1.9354 | val_acc=0.6011
Epoch 2/10 | train_loss=1.8789 | train_acc=0.5887 | val_loss=1.2183 | val_acc=0.7831
Epoch 3/10 | train_loss=1.3732 | train_acc=0.6804 | val_loss=0.7282 | val_acc=0.8395
Epoch 4/10 | train_loss=1.0379 | train_acc=0.7272 | val_loss=0.5969 | val_acc=0.8696
Epoch 5/10 | train_loss=0.9289 | train_acc=0.7579 | val_loss=0.4744 | val_acc=0.9054
Epoch 6/10 | train_loss=0.8022 | train_acc=0.7758 | val_loss=0.5982 | val_acc=0.9011
Epoch 7/10 | train_loss=0.7839 | train_acc=0.7819 | val_loss=0.4312 | val_acc=0.9197
Epoch 8/10 | train_loss=0.7170 | train_acc=0.8015 | val_loss=0.4017 | val_acc=0.9255
Epoch 9/10 | train_loss=0.7445 | train_acc=0.8025 | val_loss=0.3913 | val_acc=0.9312
Epoch 10/10 | train_loss=0.6980 | train_acc=0.8117 | val_loss=0.3882 | val_acc=0.9221
